In [0]:
%sql

-- PIPELINE: BRONZE -> SILVER
INSERT OVERWRITE catalog_ventas.silver.ventas_clean (
  venta,
  articulo,
  descrip,
  precio,
  cant,
  total,
  usulogin,
  clientes,
  vtaestado,
  vtafecha,
  turno,
  condvtapos,
  delivery,
  _source_table,
  _processing_timestamp
)

WITH datos_limpios AS (

  SELECT
  *
  FROM ventas_bronze_EDA
  WHERE precio IS NOT NULL
    AND precio > 0 AND precio <> 0.1
    AND total > 0 AND total <> 0.1
    AND articulo <> 0
    AND descrip IS NOT NULL

),
datos_duplicados AS (
  SELECT
  *,
  ROW_NUMBER() OVER(PARTITION BY 
               venta,articulo, descrip 
                ORDER BY venta) AS rn
  FROM datos_limpios
)
SELECT
  venta,
  articulo,
  descrip,
  precio,
  cant,
  total,
  usulogin,
  clientes,
  vtaestado,
  vtafecha,
  turno,
  condvtapos,
  delivery,
  'bronze_EDA' AS _source_table,
  CURRENT_TIMESTAMP AS _processing_timestamp
FROM datos_duplicados
WHERE rn = 1




In [0]:
%sql
--COMPARAR REGISTROS BRONZE VS SILVER 
SELECT 
    'Bronze (raw)' as capa,
    COUNT(*) as registros
FROM ventas_bronze_eda

UNION ALL

SELECT 
    'Silver' as capa,
    COUNT(*) as registros
FROM catalog_ventas.silver.ventas_clean;

In [0]:
%sql
--RESUMEN DE LA LIMPIEZA
SELECT
  'ventas' AS tabla,
  56839 AS bronze_registros,
  49081 AS silver_registros,
  7758 AS registros_descartados,
  ROUND(7758 / 56839 * 100, 2) AS porcentaje_limpieza